# 4.03 Cloud Storage Loaders

오브젝트 스토리지(S3, Azure Blob, GCS)와 Google Drive에서 문서를 직접 가져온다.
모두 **자격증명 키로 게이트**되기 때문에, probe가 어느 키든 하나라도 있는지 확인한 뒤 해당 경로의 셀만 실행한다.

## 학습 목표

- `S3FileLoader` / `S3DirectoryLoader`: 파일 1개 vs prefix(폴더) 대량 로드
- `AzureBlobStorageFileLoader`: 커넥션 스트링 기반 단일 blob
- `GCSFileLoader`: 서비스 계정 JSON 경로(`GOOGLE_APPLICATION_CREDENTIALS`) 사용
- `GoogleDriveLoader`: OAuth credentials → 폴더 공유 문서 적재

## 언제 쓰나

- 사내 데이터레이크 S3 prefix → RAG 코퍼스
- Azure 규제 환경에서 저장된 보고서 자동 적재
- GCS + Vertex AI 파이프라인 결합
- Google Drive 공유 폴더에 쌓이는 회의록 자동 인덱싱

## 4.03.1 환경 설정 · 키 확인

아래 probe는 네 공급자 자격증명 중 **하나라도** 있는지 확인한다. 전부 없으면 즉시 실패하며, 그 뒤 key-gated 셀은 주석 상태로 남겨둔다(각자 필요한 섹션만 주석 해제).

In [ ]:
# !pip install -U langchain-community boto3 azure-storage-blob google-cloud-storage google-auth unstructured
import os
from dotenv import load_dotenv
load_dotenv(override=True)

available = {
    "AWS":    bool(os.environ.get("AWS_ACCESS_KEY_ID")) and bool(os.environ.get("AWS_SECRET_ACCESS_KEY")),
    "Azure":  bool(os.environ.get("AZURE_STORAGE_CONNECTION_STRING")),
    "GCS":    bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")) and bool(os.environ.get("GCS_BUCKET")),
    "Drive":  bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")) and bool(os.environ.get("GOOGLE_DRIVE_FOLDER_ID")),
}
print("자격증명 상태:", available)
assert any(available.values()), (
    "AWS_ACCESS_KEY_ID / AZURE_STORAGE_CONNECTION_STRING / "
    "GOOGLE_APPLICATION_CREDENTIALS+GCS_BUCKET / "
    "GOOGLE_APPLICATION_CREDENTIALS+GOOGLE_DRIVE_FOLDER_ID 중 하나 이상 필요."
)

## 4.03.2 S3 — `S3FileLoader` · `S3DirectoryLoader`

`boto3` 기반으로 파일 단위(`S3FileLoader`) 혹은 prefix 기반 디렉터리(`S3DirectoryLoader`) 적재를 지원한다.

| 공통 인자 | 설명 |
|-----------|------|
| `bucket` | 버킷 이름 |
| `region_name` | 리전(없으면 IAM/환경변수) |
| `endpoint_url` | S3 호환 스토리지(MinIO 등) |
| `aws_access_key_id` / `_secret_access_key` | IAM 키 직주입 |
| `mode` (File만) | `single`, `elements`, `paged` — unstructured 파서 위임 |

In [ ]:
# AWS 키가 있을 때만 아래 주석을 해제
# assert available["AWS"], "AWS 자격증명 필요"
# from langchain_community.document_loaders import S3FileLoader, S3DirectoryLoader
#
# # 파일 하나
# file_loader = S3FileLoader(
#     bucket=os.environ["S3_BUCKET"],
#     key="docs/onboarding.pdf",
#     region_name=os.environ.get("AWS_REGION", "ap-northeast-2"),
# )
# docs = file_loader.load()
# print("file docs:", len(docs), "| chars:", len(docs[0].page_content))
#
# # prefix 전체
# dir_loader = S3DirectoryLoader(
#     bucket=os.environ["S3_BUCKET"],
#     prefix="docs/",
#     region_name=os.environ.get("AWS_REGION", "ap-northeast-2"),
# )
# for d in dir_loader.lazy_load():
#     print("-", d.metadata.get("source"), "|", len(d.page_content), "chars")

print("S3 섹션: 키 + S3_BUCKET 환경변수가 준비되면 위 주석 해제 후 실행.")
print("AWS 준비 상태:", available["AWS"])

## 4.03.3 Azure Blob — `AzureBlobStorageFileLoader`

커넥션 스트링 + 컨테이너 + blob 이름 3개만 있으면 된다. 컨테이너 전체를 돌 때는 `AzureBlobStorageContainerLoader`(공식 문서 경로 참고) 이용.

In [ ]:
# assert available["Azure"], "AZURE_STORAGE_CONNECTION_STRING 필요"
# from langchain_community.document_loaders import AzureBlobStorageFileLoader
#
# azure_loader = AzureBlobStorageFileLoader(
#     conn_str=os.environ["AZURE_STORAGE_CONNECTION_STRING"],
#     container=os.environ["AZURE_CONTAINER"],
#     blob_name="reports/2026-q1.pdf",
# )
# for d in azure_loader.load():
#     print(d.metadata.get("source"), "|", len(d.page_content))

print("Azure 섹션: AZURE_STORAGE_CONNECTION_STRING + AZURE_CONTAINER + blob_name 필요.")
print("Azure 준비 상태:", available["Azure"])

## 4.03.4 Google Cloud Storage — `GCSFileLoader`

GCS는 `GOOGLE_APPLICATION_CREDENTIALS`가 가리키는 서비스 계정 JSON으로 인증한다.
`loader_func`로 파싱기(예: `UnstructuredFileLoader`, `PyMuPDFLoader`)를 바꿔 PDF/HTML/PPTX를 파일 확장자에 맞게 처리할 수 있다.

In [ ]:
# assert available["GCS"], "GOOGLE_APPLICATION_CREDENTIALS + GCS_BUCKET 필요"
# from langchain_community.document_loaders import GCSFileLoader
# from langchain_community.document_loaders import PyMuPDFLoader
#
# gcs_loader = GCSFileLoader(
#     project_name=os.environ["GCP_PROJECT"],
#     bucket=os.environ["GCS_BUCKET"],
#     blob="handbook.pdf",
#     loader_func=PyMuPDFLoader,  # PDF 전용 파서 주입
# )
# for d in gcs_loader.load():
#     print(d.metadata.get("source"), "|", len(d.page_content))

print("GCS 섹션: GCP_PROJECT + GCS_BUCKET + 서비스 계정 JSON 경로 필요.")
print("GCS 준비 상태:", available["GCS"])

## 4.03.5 Google Drive — `GoogleDriveLoader`

Drive 로더는 OAuth 토큰(`token.json`) 또는 서비스 계정을 받는다.
- `folder_id`: 공유 폴더 ID (URL에서 `/folders/<id>`)
- `file_ids`: 개별 파일 ID 리스트
- `recursive=True`: 하위 폴더까지 탐색
- `file_types`: `("document", "sheet", "pdf")` 등 MIME 필터

Drive API 스코프는 최소 `https://www.googleapis.com/auth/drive.readonly` 필요.

In [ ]:
# assert available["Drive"], "GOOGLE_APPLICATION_CREDENTIALS + GOOGLE_DRIVE_FOLDER_ID 필요"
# from langchain_community.document_loaders import GoogleDriveLoader
#
# drive_loader = GoogleDriveLoader(
#     folder_id=os.environ["GOOGLE_DRIVE_FOLDER_ID"],
#     credentials_path=os.environ["GOOGLE_APPLICATION_CREDENTIALS"],
#     token_path="./drive_token.json",  # 최초 1회 브라우저 인증 후 재사용
#     recursive=False,
#     file_types=("document", "pdf"),
# )
# for d in drive_loader.load()[:3]:
#     print(d.metadata.get("source"), "|", len(d.page_content))

print("Drive 섹션: 서비스 계정 JSON + GOOGLE_DRIVE_FOLDER_ID 필요.")
print("Drive 준비 상태:", available["Drive"])

## 4.03.6 lazy_load + split·embed 파이프라인

클라우드 스토리지 로더의 가장 큰 실무 이점은 **수천 개 파일을 스트림**으로 처리할 수 있다는 점이다.
`lazy_load()` 제너레이터에 splitter + Chroma를 파이프해 메모리에 전체를 쌓지 않는다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 실제 클라우드 로더는 키가 있을 때만 구동되므로,
# 여기서는 임의 Document 리스트로 파이프라인 골격만 확인한다.
from langchain_core.documents import Document
fake_stream = iter([
    Document(page_content="클라우드 스토리지 로더는 스트림 친화적으로 설계돼 있다.", metadata={"source": "s3://demo/a.md"}),
    Document(page_content="lazy_load로 yield된 문서를 splitter에 바로 넘긴다.", metadata={"source": "s3://demo/b.md"}),
])

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
chunks = []
for d in fake_stream:
    chunks.extend(splitter.split_documents([d]))

if os.environ.get("OPENAI_API_KEY"):
    store = Chroma.from_documents(chunks, embedding=OpenAIEmbeddings(model="text-embedding-3-small"), collection_name="cloud_demo")
    for hit in store.similarity_search("스트림 처리", k=2):
        print("-", hit.metadata["source"], "|", hit.page_content)
else:
    print("OPENAI_API_KEY 없음 — split까지만 실행했다. chunks:", len(chunks))

## 체크리스트

- [ ] 자격증명은 `.env`에만 두고 노트북에 하드코딩하지 않는다
- [ ] 대량 prefix는 `lazy_load()` + splitter 스트림 파이프
- [ ] `GCSFileLoader.loader_func`로 포맷별 파서 교체(기본은 Unstructured)
- [ ] Drive는 스코프 `drive.readonly` 이상 필요, 첫 실행 시 브라우저 인증

## 다음

- `04_productivity.ipynb` — Notion/Slack/GitHub/Confluence/Figma 로더
- `../05_retrievers/` — 로더 출력 위의 retriever 패턴

## 참고

- S3 로더: https://python.langchain.com/docs/integrations/document_loaders/aws_s3_file/
- Azure Blob: https://python.langchain.com/docs/integrations/document_loaders/azure_blob_storage_file/
- GCS: https://python.langchain.com/docs/integrations/document_loaders/google_cloud_storage_file/
- Google Drive: https://python.langchain.com/docs/integrations/document_loaders/google_drive/